In [ ]:
# Filename: process_legal.py
# pip install tiktoken sentencepiece protobuf
# ---------------------------------------------------------
# 1. CONFIGURATION
# 2. HELPERS
#   a. loading stopwords
#   b. loading Thesaurus
#   c. verify/validate quantization before model load
#   d. model load from previous model save, if fails continue from scratch
#   e. model unload (explicit RAM/VRAM release)
# 3. RAG MANAGER
# 4. PIPELINE  — three sequential phases, one model resident at a time
#   a. Phase 1 – Ingest: read all files, build RAG (sentence-transformer only)
#   b. Phase 2 – Summarise: CPU summariser → summarise all → unload
#   c. Phase 3 – Generate: CUDA generator → RAG-retrieve + generate → unload
# 5. MAIN FUNCTION
# ---------------------------------------------------------

import gc, os, re, torch, psutil
torch.set_num_threads(24)

USE_CUDA = torch.cuda.is_available()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True,"
    "max_split_size_mb:128,"
    "garbage_collection_threshold:0.8"
)

USE_CUDA = torch.cuda.is_available()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True,"
    "max_split_size_mb:128,"
    "garbage_collection_threshold:0.8"
)

if USE_CUDA:
    # if using Windows Native change to False
    torch.backends.cuda.enable_flash_sdp(False)
    # if using Windows Native change to False
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    # if using Windows Native change to True
    torch.backends.cuda.enable_math_sdp(True)

import pandas as pd
import psutil
from datetime import datetime
from typing import Callable, List, Set

from langchain_community.document_loaders import WebBaseLoader, Docx2txtLoader, TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from striprtf.striprtf import rtf_to_text
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

from textual.app import App, ComposeResult
from textual.widgets import Header, Footer, RichLog, ProgressBar, DataTable, Static, Rule
from textual.containers import Horizontal, Vertical
from textual import work

# =============================================================================
# 1. CONFIGURATION
# =============================================================================
CONFIG = {
    "mode": "title",          # "keyword" or "title"
    #"main_model_id": "Qwen/Qwen3-4B-Thinking-2507",
    #"main_model_id": "nvidia/Nemotron-Terminal-8B",
    "main_model_id": "unsloth/Qwen3.5-2B",
    "summ_model_id": "slovak-nlp/mistral-sk-7b",
    #"summ_model_id": "slovak-nlp/Qwen3-14B-sk",
    "emb_model_id":  "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    "input_dir":       "./Input",
    "output_dir":      "./Output",
    "models_dir":      "./Models",
    "rag_dir":         "./RAG_store",
    "thesaurus_path":  "./Thesaurus/SK_Local_Thesaurus.xlsx",
    "thesaurus_col":   "slovak_terms",
    "stopwords_path":  "./Input/stopword_dictionary.txt",
    "allowed_domains": ["slov-lex.sk", "justice.gov.sk", "najpravo.sk", "zakonypreludi.sk"],
    "chunk_size":    1200,
    "chunk_overlap":  200,
    "top_k":            4,
}

for d in (CONFIG["output_dir"], CONFIG["rag_dir"], CONFIG["models_dir"]):
    os.makedirs(d, exist_ok=True)

# Aliases used by the TUI phase tracker
_LOG   = Callable[[str], None]          # log_fn signature
_PROG  = Callable[[int, int], None]     # progress_fn(current, total)
_RES   = Callable[[dict], None]         # result_fn(row_dict)

# =============================================================================
# 2. HELPERS
# =============================================================================

def load_rtf(path: str) -> List[Document]:
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            return [Document(page_content=rtf_to_text(f.read()), metadata={"source": path})]
    except Exception as e:
        return []

# a. Stopwords
def load_stopwords(path: str) -> Set[str]:
    if not os.path.exists(path):
        return set()
    with open(path, "r", encoding="utf-8") as f:
        return {line.strip().lower() for line in f if line.strip()}

# b. Thesaurus
def load_thesaurus(
    path: str,
    col: str,
    stopwords: Set[str],
    log_fn: _LOG = print,
) -> List[str]:
    if not os.path.exists(path):
        return []
    df = pd.read_excel(path, engine="openpyxl")
    if col not in df.columns:
        return []
    seen, terms = set(), []
    for cell in df[col].dropna().astype(str):
        for t in re.split(r"[,\n;]+", cell):
            t = t.strip()
            if len(t) > 1 and t.lower() not in stopwords and t not in seen:
                seen.add(t)
                terms.append(t)
    log_fn(f"[THESAURUS] Loaded [bold]{len(terms)}[/bold] terms.")
    return terms

def match_terms(text: str, terms: List[str], limit: int = 10) -> str:
    if not text or not terms:
        return ""
    hits = [t for t in terms if re.search(rf"\b{re.escape(t)}\b", text, re.IGNORECASE)]
    hits.sort(key=len, reverse=True)
    return ", ".join(hits[:limit])

# c. Quantization validation
def _verify_quantization(
    quantize: str | None,
    device: str,
    log_fn: _LOG = print,
) -> str | None:
    if quantize is None:
        return None

    if device == "cpu":
        log_fn(f"Quantization '{quantize}' not supported on CPU → fallback to fp32.")
        return None

    if not torch.cuda.is_available():
        log_fn("CUDA not available → fallback to fp32.")
        return None

    try:
        import bitsandbytes as bnb
        log_fn(f"Bitsandbytes [bold]{bnb.__version__}[/bold] detected.")
    except ImportError:
        log_fn("[yellow]Bitsandbytes not installed → fallback to fp16.[/yellow]")
        return None
    except Exception as e:
        log_fn(f"[yellow]Bitsandbytes import failed: {e} → fallback to fp16.[/yellow]")
        return None

    try:
        if quantize == "4bit":
            _ = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
            )
        elif quantize == "8bit":
            _ = BitsAndBytesConfig(load_in_8bit=True)
        else:
            log_fn(f"[yellow]Unknown quantization '{quantize}' → fallback to fp16.[/yellow]")
            return None
    except Exception as e:
        log_fn(f"[yellow]BitsAndBytesConfig init failed: {e} → fallback to fp16.[/yellow]")
        return None

    try:
        free_mem, total_mem = torch.cuda.mem_get_info()
        free_gb = free_mem / (1024 ** 3)
        log_fn(f"CUDA free VRAM: [bold]{free_gb:.2f} GB[/bold]")
        if free_gb < 2.0:
            log_fn(f"[yellow]Very low VRAM ({free_gb:.2f} GB) — quantization may fail.[/yellow]")
    except Exception:
        pass

    log_fn(f"Quantization verification OK for [bold]{quantize!r}[/bold].")
    return quantize

# d. Model load (with local cache)
def load_model(
    model_id: str,
    device: str,
    quantize: str | None = None,
    log_fn: _LOG = print,
):
    quantize = _verify_quantization(quantize, device, log_fn=log_fn)

    if quantize in ("4bit", "8bit"):
        variant = quantize
    elif device.startswith("cuda"):
        variant = "fp16"
    else:
        variant = "fp32"

    safe_name  = model_id.replace("/", "_")
    local_path = os.path.join(CONFIG["models_dir"], f"{safe_name}__{variant}")

    has_config  = os.path.exists(os.path.join(local_path, "config.json"))
    weight_files = (
        "model.safetensors",
        "pytorch_model.bin",
        "model.safetensors.index.json",
        "pytorch_model.bin.index.json",
    )
    has_weights = any(os.path.exists(os.path.join(local_path, w)) for w in weight_files)
    from_local  = has_config and has_weights

    if from_local:
        source = local_path
        log_fn(f"[{device.upper()}] Loading [bold]{model_id}[/bold] from LOCAL cache ({variant})")
    else:
        source = model_id
        log_fn(f"[{device.upper()}] Loading [bold]{model_id}[/bold] from HuggingFace ({variant})")

    bnb_config = None
    if quantize == "4bit":
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
    elif quantize == "8bit":
        bnb_config = BitsAndBytesConfig(load_in_8bit=True)

    kwargs = dict(device_map=device, trust_remote_code=True, low_cpu_mem_usage=True)

    if from_local:
        if bnb_config is None:
            kwargs["dtype"] = torch.float16 if device.startswith("cuda") else torch.float32
    else:
        if bnb_config is not None:
            kwargs["quantization_config"] = bnb_config
        else:
            kwargs["dtype"] = torch.float16 if device.startswith("cuda") else torch.float32

    try:
        tok = AutoTokenizer.from_pretrained(source, trust_remote_code=True)

        try:
            model = AutoModelForCausalLM.from_pretrained(
                source, attn_implementation="sdpa", **kwargs
            )
        except Exception as e:
            if bnb_config is not None:
                log_fn(f"[yellow][QUANT] Quantized load failed: {e}[/yellow]")
                log_fn("[yellow][QUANT] Retrying without quantization (fp16)…[/yellow]")
                kwargs.pop("quantization_config", None)
                kwargs["dtype"] = torch.float16 if device.startswith("cuda") else torch.float32
                model = AutoModelForCausalLM.from_pretrained(source, **kwargs)
                bnb_config = None
                variant    = "fp16"
                local_path = os.path.join(CONFIG["models_dir"], f"{safe_name}__{variant}")
            else:
                raise

        if not from_local:
            try:
                os.makedirs(local_path, exist_ok=True)
                tok.save_pretrained(local_path)
                model.save_pretrained(local_path, safe_serialization=True)
                log_fn(f"[dim]Model ({variant}) cached to {local_path}[/dim]")
            except Exception as e:
                log_fn(f"[yellow]Could not save model locally ({variant}): {e}[/yellow]")
                try:
                    tok.save_pretrained(local_path)
                    log_fn(f"[dim]Tokenizer-only cached to {local_path}[/dim]")
                except Exception as e2:
                    log_fn(f"[yellow]Could not save tokenizer: {e2}[/yellow]")

        return pipeline("text-generation", model=model, tokenizer=tok)

    except Exception as e:
        log_fn(f"[bold red]Load failed ({device}, {quantize}): {e}[/bold red]")
        return None

def run_llm(pipe, prompt: str, split_on: str) -> str:
    try:
        res = pipe(prompt)
        return res[0]["generated_text"].split(split_on)[-1].strip()
    except Exception as e:
        return ""

# e. Model unload — returns None so caller can clear their reference
def unload_model(pipe, log_fn: _LOG = print):
    if pipe is None:
        return None
    try:
        del pipe.model
        del pipe.tokenizer
    except AttributeError:
        pass
    del pipe
    gc.collect()
    if USE_CUDA:
        torch.cuda.empty_cache()
    log_fn("[dim][MEM] Model unloaded — memory freed.[/dim]")
    return None

# =============================================================================
# 3. RAG MANAGER
# =============================================================================
class RAGManager:
    def __init__(self, log_fn: _LOG = print):
        self._log = log_fn
        log_fn("Loading embeddings on CPU…")
        self.embeddings = HuggingFaceEmbeddings(
            model_name=CONFIG["emb_model_id"],
            model_kwargs={"device": "cpu"},
        )
        self.vs = Chroma(
            collection_name="sk_legal_docs",
            persist_directory=CONFIG["rag_dir"],
            embedding_function=self.embeddings,
        )
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=CONFIG["chunk_size"],
            chunk_overlap=CONFIG["chunk_overlap"],
        )

    def add_text(self, text: str, metadata: dict):
        docs = self.splitter.create_documents([text], metadatas=[metadata])
        if docs:
            self.vs.add_documents(docs)

    def search(self, query: str) -> str:
        docs = self.vs.similarity_search(query, k=CONFIG["top_k"])
        return "\n\n".join(d.page_content for d in docs)

    def ingest_web(self, urls: List[str]):
        valid = [u for u in urls if any(d in u for d in CONFIG["allowed_domains"])]
        if not valid:
            return
        try:
            docs = self.splitter.split_documents(WebBaseLoader(valid).load())
            self.vs.add_documents(docs)
            self._log(f"[WEB] Ingested [bold]{len(docs)}[/bold] chunks from {len(valid)} URL(s).")
        except Exception as e:
            self._log(f"[yellow][WEB] Ingest failed: {e}[/yellow]")

# =============================================================================
# 4. PIPELINE
# =============================================================================

LOADERS = {
    ".docx": Docx2txtLoader,
    ".rtf":  load_rtf,
    ".txt":  lambda p: TextLoader(p, encoding="utf-8"),
}

def read_file(path: str) -> str:
    ext    = os.path.splitext(path)[1].lower()
    loader = LOADERS.get(ext)
    if loader is None:
        return ""
    try:
        docs = loader(path).load() if ext != ".rtf" else loader(path)
        return "\n".join(d.page_content for d in docs)
    except Exception as e:
        return ""

# a. Phase 1 – Ingest (sentence-transformer only)
def phase_ingest(
    files: List[str],
    rag: RAGManager,
    log_fn: _LOG = print,
    progress_fn: _PROG = None,
) -> dict:
    corpus = {}
    total  = len(files)
    for idx, filepath in enumerate(files, 1):
        filename = os.path.basename(filepath)
        log_fn(f"[INGEST] [bold]{filename}[/bold]")
        text = read_file(filepath)
        if not text.strip():
            log_fn(f"  [dim]→ empty, skipping.[/dim]")
        else:
            rag.add_text(text, {"file": filename})
            corpus[filepath] = text
        if progress_fn:
            progress_fn(idx, total)
    log_fn(f"[INGEST] [bold green]Done[/bold green] — {len(corpus)} of {total} files indexed.")
    return corpus

# b. Phase 2 – Summarise (CPU model)
def phase_summarise(
    corpus: dict,
    summarizer,
    log_fn: _LOG = print,
    progress_fn: _PROG = None,
) -> dict:
    summaries = {}
    total     = len(corpus)
    for idx, (filepath, text) in enumerate(corpus.items(), 1):
        filename = os.path.basename(filepath)
        log_fn(f"[SUMM] [bold]{filename}[/bold]")
        prompt = (
            f"Zhrň nasledovný text do stručného odstavca (max 5 viet). Píš po slovensky.\n\n"
            f"TEXT:\n{text[:12000]}\n\nZHRNUTIE:"
        )
        summary = run_llm(summarizer, prompt, "ZHRNUTIE:") or text[:500]
        summaries[filepath] = summary
        if progress_fn:
            progress_fn(idx, total)
    log_fn(f"[SUMM] [bold green]Done[/bold green] — {len(summaries)} summaries produced.")
    return summaries

# c. Phase 3 – Generate (CUDA model)
def phase_generate(
    corpus: dict,
    summaries: dict,
    rag: RAGManager,
    generator,
    terms: List[str],
    log_fn: _LOG = print,
    progress_fn: _PROG = None,
    result_fn: _RES = None,
) -> List[dict]:
    mode = CONFIG["mode"]
    prompt_instruction = (
        "Navrhni JEDEN presný právny nadpis (Title)"
        if mode == "title"
        else "Vyber JEDEN kľúčový pojem (Keyword)"
    )
    suffix = "Nadpis:" if mode == "title" else "Kľúčový pojem:"

    results = []
    total   = len(corpus)
    for idx, (filepath, text) in enumerate(corpus.items(), 1):
        filename = os.path.basename(filepath)
        log_fn(f"[GEN] [bold]{filename}[/bold]")
        summary = summaries.get(filepath, text[:500])
        matched = match_terms(text, terms)
        ctx     = rag.search(f"{summary} {matched}")

        gen_prompt = (
            f"Kontext:\n{ctx[:2000]}\n\n"
            f"Zadanie: Na základe súhrnu a pojmov ({matched}) {prompt_instruction}.\n\n"
            f"Súhrn:\n{summary}\n\n{suffix}"
        )
        raw    = run_llm(generator, gen_prompt, suffix)
        output = raw.split("\n")[0].strip('" ') if raw else "ERROR"

        row = {
            "file":             filename,
            "mode":             mode,
            "summary":          summary,
            "priority_terms":   matched,
            "generated_output": output,
        }
        results.append(row)
        if result_fn:
            result_fn(row)
        if progress_fn:
            progress_fn(idx, total)

    log_fn(f"[GEN] [bold green]Done[/bold green] — {len(results)} outputs generated.")
    return results


# =============================================================================
# 5. TEXTUAL TUI
# =============================================================================

_PHASE_ICON = {"pending": "○", "active": "◉", "done": "✓"}
_PHASE_COLOR = {
    "pending": "dim",
    "active":  "bold yellow",
    "done":    "bold green",
}
_PHASE_LABELS = [
    "Phase 1 — Ingest  (sentence-transformer)",
    "Phase 2 — Summarise  (CPU model)",
    "Phase 3 — Generate   (CUDA model)",
]

class LegalPipelineApp(App):
    TITLE   = "SK Legal Pipeline"
    SUB_TITLE = f"Mode: {CONFIG['mode'].upper()}  |  RTX 4070 Super"

    CSS = """
    Screen {
        background: $background;
    }

    #main {
        height: 1fr;
        margin-bottom: 1;
    }

    #left {
        width: 40;
        border: solid $primary-darken-2;
        padding: 0 1;
        margin-right: 1;
    }

    #right {
        border: solid $primary-darken-2;
        padding: 0 1;
    }

    #results {
        height: 13;
        border: solid $accent-darken-1;
        padding: 0 1;
    }

    RichLog {
        height: 1fr;
        scrollbar-size: 1 1;
    }

    DataTable {
        height: 1fr;
    }

    .section {
        color: $accent;
        text-style: bold;
        padding: 1 0 0 0;
    }

    ProgressBar {
        margin: 0 0 0 0;
    }

    #progress-text {
        color: $foreground 60%;
        margin-bottom: 1;
    }

    #vram-stat, #ram-stat {
        color: $foreground 55%;
    }
    """

    BINDINGS = [("q", "quit", "Quit")]

    def compose(self) -> ComposeResult:
        yield Header()

        with Horizontal(id="main"):
            with Vertical(id="left"):
                yield Static("── PHASES ──", classes="section")
                for i, label in enumerate(_PHASE_LABELS, 1):
                    yield Static(
                        f"[dim]{_PHASE_ICON['pending']} {label}[/dim]",
                        id=f"phase-{i}",
                    )

                yield Rule()
                yield Static("── PROGRESS ──", classes="section")
                yield Static("Waiting to start…", id="phase-label")
                yield ProgressBar(total=100, show_eta=False, id="prog")
                yield Static("0 / 0 files", id="progress-text")

                yield Rule()
                yield Static("── MEMORY ──", classes="section")
                yield Static("VRAM: —", id="vram-stat")
                yield Static("RAM:  —", id="ram-stat")

            with Vertical(id="right"):
                yield Static("── LOG ──", classes="section")
                yield RichLog(id="log", highlight=True, markup=True, wrap=True)

        with Vertical(id="results"):
            yield Static("── RESULTS ──", classes="section")
            yield DataTable(id="table", show_cursor=False)

        yield Footer()

    def on_mount(self) -> None:
        tbl = self.query_one("#table", DataTable)
        tbl.add_columns("File", "Generated Output", "Matched Terms")
        self.set_interval(3.0, self._refresh_memory)
        self.run_pipeline()

    def _write_log(self, msg: str) -> None:
        self.query_one("#log", RichLog).write(msg)

    def _set_phase(self, num: int, state: str) -> None:
        icon  = _PHASE_ICON[state]
        color = _PHASE_COLOR[state]
        label = _PHASE_LABELS[num - 1]
        self.query_one(f"#phase-{num}", Static).update(
            f"[{color}]{icon} {label}[/{color}]"
        )

    def _set_progress(self, current: int, total: int, label: str = "") -> None:
        pct = int(current / total * 100) if total else 0
        self.query_one("#prog", ProgressBar).update(progress=pct)
        self.query_one("#progress-text", Static).update(f"{current} / {total} files")
        if label:
            self.query_one("#phase-label", Static).update(label)

    def _add_result(self, row: dict) -> None:
        tbl = self.query_one("#table", DataTable)
        terms_preview = (row["priority_terms"] or "—")[:45]
        tbl.add_row(row["file"], row["generated_output"], terms_preview)

    def _refresh_memory(self) -> None:
        # RAM
        ram_gb = psutil.Process().memory_info().rss / (1024 ** 3)
        self.query_one("#ram-stat", Static).update(f"RAM:  {ram_gb:.1f} GB")
        # VRAM
        if USE_CUDA:
            try:
                free, total = torch.cuda.mem_get_info()
                used_gb  = (total - free) / (1024 ** 3)
                total_gb = total / (1024 ** 3)
                self.query_one("#vram-stat", Static).update(
                    f"VRAM: {used_gb:.1f} / {total_gb:.1f} GB"
                )
            except Exception:
                pass
        else:
            self.query_one("#vram-stat", Static).update("VRAM: N/A (CPU only)")

    @work(thread=True)
    def run_pipeline(self) -> None:
        # Thread-safe helpers
        def log(msg: str) -> None:
            self.call_from_thread(self._write_log, msg)

        def set_phase(n: int, state: str) -> None:
            self.call_from_thread(self._set_phase, n, state)

        def set_progress(current: int, total: int, label: str = "") -> None:
            self.call_from_thread(self._set_progress, current, total, label)

        def add_result(row: dict) -> None:
            self.call_from_thread(self._add_result, row)

        try:
            if USE_CUDA:
                torch.cuda.empty_cache()

            log(f"[bold cyan]▶ Pipeline started — mode: {CONFIG['mode']}[/bold cyan]")

            stopwords = load_stopwords(CONFIG["stopwords_path"])
            terms     = load_thesaurus(
                CONFIG["thesaurus_path"], CONFIG["thesaurus_col"], stopwords, log_fn=log
            )

            files = [
                os.path.join(CONFIG["input_dir"], f)
                for f in os.listdir(CONFIG["input_dir"])
                if f.lower().endswith((".docx", ".rtf", ".txt"))
            ]
            log(f"Found [bold]{len(files)}[/bold] input file(s).")

            log("")
            log("[bold]━━ PHASE 1  Ingest ━━[/bold]")
            set_phase(1, "active")
            set_progress(0, len(files), "Ingesting files…")

            rag = RAGManager(log_fn=log)
            rag.ingest_web(["https://www.slov-lex.sk/pravne-predpisy/SK/ZZ/2005/300/"])
            corpus = phase_ingest(
                files, rag, log_fn=log,
                progress_fn=lambda c, t: set_progress(c, t, "Ingesting…"),
            )
            set_phase(1, "done")

            if not corpus:
                log("[bold red]No processable files found. Exiting.[/bold red]")
                return

            log("")
            log("[bold]━━ PHASE 2  Summarise ━━[/bold]")
            set_phase(2, "active")
            set_progress(0, len(corpus), "Loading summariser…")

            summarizer = load_model(CONFIG["summ_model_id"], "cpu", log_fn=log)
            if not summarizer:
                log("[bold red]Summariser failed to load.[/bold red]")
                return

            summaries = phase_summarise(
                corpus, summarizer, log_fn=log,
                progress_fn=lambda c, t: set_progress(c, t, "Summarising…"),
            )
            summarizer = unload_model(summarizer, log_fn=log)   # frees ~28 GB RAM
            set_phase(2, "done")

            log("")
            log("[bold]━━ PHASE 3  Generate ━━[/bold]")
            set_phase(3, "active")
            set_progress(0, len(corpus), "Loading generator…")

            generator = load_model(
                CONFIG["main_model_id"], "cuda:0", quantize="8bit", log_fn=log
            )
            if not generator:
                log("[bold red]Generator failed to load.[/bold red]")
                return

            results = phase_generate(
                corpus, summaries, rag, generator, terms,
                log_fn=log,
                progress_fn=lambda c, t: set_progress(c, t, "Generating…"),
                result_fn=add_result,
            )
            generator = unload_model(generator, log_fn=log)     # returns VRAM
            set_phase(3, "done")

            if results:
                df       = pd.DataFrame(results)
                out_name = f"output_{CONFIG['mode']}_{datetime.now():%Y-%m-%d}.csv"
                df.to_csv(
                    os.path.join(CONFIG["output_dir"], out_name),
                    index=False,
                    encoding="utf-8-sig",
                )
                log("")
                log(f"[bold green]✓ Saved → {out_name}[/bold green]")

            set_progress(len(corpus), len(corpus), "Complete ✓")
            log("[bold green]▶ Pipeline complete.[/bold green]")

        except Exception as exc:
            import traceback
            log(f"[bold red]Pipeline error: {exc}[/bold red]")
            log(traceback.format_exc())

        finally:
            if USE_CUDA:
                torch.cuda.empty_cache()
#==========================================================================
# 6. ENTRY POINT
# =============================================================================
if __name__ == "__main__":
    LegalPipelineApp().run()